# 114 — Ciclo ReAct y observación del entorno

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**ReAct** (Yao et al., arXiv:2210.03629) estructura cada iteración del agente en tres
elementos: **thought** (razonamiento en texto, sin efectos), **action** (invocación de
una herramienta con argumentos, o `finish`) y **observation** (la respuesta REAL del
entorno — el modelo no la genera: la recibe).

```text
repetir hasta finish o presupuesto agotado:
    thought_t     = LLM(contexto)             # razonar sobre lo observado
    action_t      = LLM(contexto + thought)   # decidir tool + args
    observation_t = entorno.ejecutar(action)  # información nueva y verificada
    contexto     += thought + action + observation
```

La observación es la única entrada de información nueva al bucle: sin ella el modelo
solo puede alucinar el estado del mundo (modo chain-of-thought). Los errores también
son observaciones — y de las más valiosas para el siguiente thought.

### 🔄 Lo que garantiza el patrón (bien implementado)

1. **Grounding:** cada decisión se toma sobre el último estado observado.
2. **Traza auditable:** la secuencia `(thought, action, observation)*` explica el porqué
   de cada paso — depurar un agente es leer su traza.
3. **Parada:** por decisión (`finish` con condiciones verificadas) o por presupuesto de
   pasos; nunca por "ya ejecuté lo que tenía pensado".

El laboratorio `agent` emite una traza de dos acciones (`status()` → obs
`{"healthy": true}`, `sum(7,5)` → obs `12`) con los thoughts implícitos: la condición
de éxito (`healthy == true` y `sum == 12`) se evalúa contra observaciones.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** En cada paso, `action` (tool + args) la genera la **política** de
decisión (aquí cableada; en un agente LLM, el modelo), y `observation` la genera el
**entorno** al ejecutar la herramienta. Los thoughts irían entre medias: antes de cada
action, como texto que razona sobre las observaciones acumuladas. Si un mismo
componente escribiera action y observation, la traza dejaría de ser evidencia: el
agente podría "observar" lo que le conviene.

**Ejercicio 2.** Thoughts válidos: (1) "No tengo ningún hecho verificado; el objetivo
exige estado y suma. Empiezo por `status()`". (2) "Obs 1 verifica `healthy == true`.
Falta la suma; la calculo con la herramienta, no de memoria". (3) "healthy ✓ (Obs 1) y
sum == 12 ✓ (Obs 2): ambas condiciones ancladas a observaciones; puedo terminar".
La clave: ningún thought afirma hechos aún no observados.

**Ejercicio 3.** Con fallo inicial: T1 → `status()` → `{"healthy": false}`; T2 reacciona
("el servicio no está sano; reintento o diagnostico") → `status()` →
`{"healthy": true}`; T3 → `sum(7,5)` → `12`; T4 → `finish`. Consume 3 acciones + finish.
`finish` en éxito solo es legítimo tras observar `healthy: true`: la primera observación
NO queda invalidada por deseo, sino por una observación posterior.

**Ejercicio 4.** (a) CoT puede razonar "Mary Shelley… estudió en X" e inventar la
universidad o el año: no tiene forma de verificar. (b) Act-only puede buscar "autora de
Frankenstein" y luego no saber componer la segunda búsqueda (¿universidad? ¿año?) porque
no articula el sub-objetivo. (c) ReAct razona el plan (persona → institución → año) y
ancla cada salto a una búsqueda; puede aún fallar si un thought elige mal la consulta,
pero cada afirmación intermedia es contrastable con su observación.

In [ ]:
result = run_lab("agent", seed=114)
assert result["kind"] == "agent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — clasificación de campos, verificada
result = run_lab("agent", seed=114)
for i, paso in enumerate(result["result"]["trace"], 1):
    print(f"Paso {i}:")
    print("  action      -> POLITICA :", paso["action"])
    print("  observation -> ENTORNO  :", paso["observation"])
# En ReAct explícito, cada paso tendría además un campo 'thought' (texto del modelo)
# generado ANTES de la action y basado solo en observaciones previas.


In [ ]:
# Ejercicio 3 — traza contrafactual con reintento, en formato auditable
traza_contrafactual = [
    {"thought": "Sin hechos verificados; verifico el estado primero.",
     "action": {"tool": "status", "args": {}},
     "observation": {"service": "demo", "healthy": False}},
    {"thought": "healthy == false observado; reintento antes de abortar.",
     "action": {"tool": "status", "args": {}},
     "observation": {"service": "demo", "healthy": True}},
    {"thought": "healthy verificado en el reintento; falta la suma.",
     "action": {"tool": "sum", "args": {"left": 7, "right": 5}},
     "observation": 12},
    {"thought": "Ambas condiciones ancladas a observaciones; termino.",
     "action": {"tool": "finish", "args": {"healthy": True, "sum": 12}},
     "observation": None},
]
for p in traza_contrafactual:
    print(p["thought"], "->", p["action"]["tool"])


## Reflexión

1. En la traza del laboratorio, ¿qué campo de cada paso proviene del entorno y cuál
   proviene de la política de decisión? ¿Por qué el grounding desaparecería si el mismo
   componente pudiera escribir ambos?
2. ReAct reduce la alucinación frente a chain-of-thought puro, pero el paper reporta
   errores de *reasoning* aun con observaciones correctas. ¿Qué aspecto de un thought
   temprano puede sesgar toda la trayectoria y qué mecanismo de la clase 115 lo mitiga?
3. Si `sum(7, 5)` devolviera `{"error": "timeout"}`, ¿qué debería contener el siguiente
   thought y por qué silenciar ese error en la herramienta "ciega" al agente?